In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import torch

from helpers.data_management import (DatasetConfig,
    create_LOSO_dataset_dataloader, save_checkpoint)
from helpers.constants import *
from helpers.visualization import inspect_knee_moment_dataset, plot_loss, prediction_overlay
from helpers.modules import KneeCNN
from helpers.running import train_model, evaluate_model, loso_cross_validation

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')


# Example Dataset

In [ ]:
leave_one_out_subject = SUBJECTS[2]
batch_size = 32

cfg = DatasetConfig(tasks = PERIODIC_TASK_PREFIXES)
train_dataset, test_dataset, train_dataloader, test_dataloader = \
    create_LOSO_dataset_dataloader(
        leave_one_out_subject, dataset_cfg=cfg, batch_size=batch_size)

print(len(train_dataset))
print(len(test_dataset))

iterator = iter(train_dataset) # only for visualization

In [ ]:
# visualize dataset
x,y = next(iterator)
_,_ = inspect_knee_moment_dataset(torch.squeeze(x),y)


In [ ]:
cnn_model = KneeCNN(in_channels=14).to(device)
checkpoint = train_model(cnn_model, train_dataloader, test_dataloader, num_epochs=10)

plot_loss(checkpoint['train_losses'], checkpoint['test_losses'])
save_checkpoint(checkpoint, 'cnn_model')

In [ ]:
preds, targets, rmse, r2 = evaluate_model(cnn_model, test_dataloader, train_dataset.scaler_y)
print(f'RMSE: {rmse:.4f} Nm')
print(f'R\u00b2:   {r2:.4f}')

prediction_overlay(targets, preds, rmse, r2)

# LOSO Evaluation

In [ ]:
cfg = DatasetConfig(tasks = ["normal_walk"], dataset_folder = "ProcessedDataTrimmed")

loso_cnn = KneeCNN(in_channels=14).to(device)

rmses, r2s = loso_cross_validation(SUBJECTS, loso_cnn, dataset_cfg = cfg, num_epoches = 5)

In [ ]:
print(rmses)
print(r2s)

# Ablation Studies

In [ ]:
ablated_sensor = "imu_sim"

cfg = DatasetConfig(tasks = ["normal_walk"], ablated_sensor = ablated_sensor, dataset_folder = "ProcessedDataTrimmed")


rmses, r2s = loso_cross_validation(SUBJECTS, loso_cnn, dataset_cfg = cfg, num_epoches = 5)
